# triloop analysis notebook

Three-loop magnetic-field array polarimetry on a single HDF5 capture.

Edit the `loops_config` cell (cell 2) and the analysis-parameters cell (cell 3) for your array geometry and target carrier.

In [ ]:
# 1) Load capture
import numpy as np
import matplotlib.pyplot as plt
from triloop import (read_capture, analyze, beamform_grid,
                      default_loops_config)

INPUT_FILE = "__INPUT_FILE_PATH__"
cap = read_capture(INPUT_FILE)
print(f"sample_rate: {cap['sample_rate']:.4g} Hz")
print(f"duration:    {cap['duration_s']:.4g} s")
print(f"channels:    {list(cap['channels'].keys())}")
print(f"scope:       {cap['scope_model']}")
t = cap['time']

In [ ]:
# 2) Loop geometry config — edit this cell to match YOUR array.
#    'channel' must match the channel names present in cap['channels'].
#    normal_az_deg = horizontal-projection azimuth of loop normal,
#                    measured CW from N (so 0 = N, 90 = E)
#    normal_el_deg = elevation of normal above horizontal
#    gain          = scalar amplitude calibration (V per unit B)
#    phase_offset_deg = relative electrical phase calibration at f0
loops_config = default_loops_config()
if cap.get('loops_config'):
    print('using loops_config from capture metadata')
    loops_config = cap['loops_config']
else:
    print('using default cube-vertex layout')
loops_config

In [ ]:
# 3) Analysis parameters — edit for your target carrier and direction guess
carrier_hz = 25_000.0     # target IF carrier (Hz)
bw_hz      = 2000.0       # ± analysis bandwidth (kept window after mixing)
az_deg     = 273.0        # initial guess azimuth (deg from N CW)
el_deg     = 12.0         # initial guess elevation (deg above horizon)
loop_channels = [lp['channel'] for lp in loops_config['loops']]
B1, B2, B3 = (cap['channels'][c] for c in loop_channels)

In [ ]:
# 4) Raw time-series and spectra
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
n_show = min(int(0.020 * cap['sample_rate']), len(t))
for B, lab, c in zip([B1, B2, B3], loop_channels, ['tab:blue','tab:green','tab:orange']):
    axes[0].plot(t[:n_show]*1e3, B[:n_show], lw=0.5, color=c, label=f"{lab}")
axes[0].set_xlabel('time (ms)'); axes[0].set_ylabel('signal'); axes[0].grid(True, alpha=0.3); axes[0].legend()
axes[0].set_title('First 20 ms of raw loop signals')
for B, lab, c in zip([B1, B2, B3], loop_channels, ['tab:blue','tab:green','tab:orange']):
    F = np.fft.rfft(B); f = np.fft.rfftfreq(len(B), 1.0/cap['sample_rate'])
    db = 20*np.log10(np.abs(F)+1e-30); db -= db.max()
    axes[1].plot(f, db, lw=0.5, color=c, label=f"{lab}")
axes[1].set_xlim(0, min(cap['sample_rate']/2, carrier_hz*4))
axes[1].axvline(carrier_hz, color='red', ls='--', lw=0.5, label=f'target {carrier_hz:.0f} Hz')
axes[1].set_xlabel('frequency (Hz)'); axes[1].set_ylabel('|FFT| (dB, peak-norm)')
axes[1].set_ylim(-80, 5); axes[1].grid(True, alpha=0.3); axes[1].legend()
fig.tight_layout()

In [ ]:
# 5) Run the analysis pipeline
res = analyze(t, B1, B2, B3, carrier_hz, bw_hz, az_deg, el_deg,
              loops_config=loops_config)
print(f'detected f_peak  : {res.f_peak:.4f} Hz')
print(f'SNR per loop (dB): {res.snr_db_per_loop}')
print(f'median pol_frac  : {np.median(res.pol_fraction):.4f}')
print(f'median ellipticity: {np.median(res.ellipticity_deg):+.2f} deg  (0=linear, ±45=circular)')
print(f'median position angle: {np.median(res.position_angle_deg):+.2f} deg (relative to p̂)')

In [ ]:
# 6) Time-resolved diagnostics
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
axes[0].plot(t, np.sqrt(res.intensity), lw=0.6, color='tab:blue', label='|B_⊥(t)|  pol-indep')
axes[0].plot(t, res.amp_dominant,        lw=0.6, color='tab:red', alpha=0.7, label='dominant pol')
axes[0].set_ylabel('amplitude'); axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot(t, np.rad2deg(res.instant_phase), lw=0.6, color='tab:purple')
axes[1].set_ylabel('phase residual (deg)'); axes[1].grid(True, alpha=0.3)

axes[2].plot(t, res.instant_freq - res.f_peak, lw=0.6, color='tab:green')
axes[2].set_ylabel(f'inst freq − f_peak (Hz)'); axes[2].grid(True, alpha=0.3)

axes[3].plot(t, res.ellipticity_deg, lw=0.6, color='tab:orange', label='ellipticity (deg)')
axes[3].plot(t, res.position_angle_deg, lw=0.6, color='tab:cyan', label='position angle (deg)')
axes[3].plot(t, 100*res.pol_fraction, lw=0.6, color='black', alpha=0.5, label='pol fraction (%)')
axes[3].set_ylabel('polarization'); axes[3].set_xlabel('time (s)')
axes[3].grid(True, alpha=0.3); axes[3].legend()
fig.tight_layout()

In [ ]:
# 7) Beamforming sweep — direction-finding via P(az, el) maximum
P, az_grid, el_grid = beamform_grid(res.z_loops, loops_config,
                                     az_grid_deg=np.arange(0,360,3),
                                     el_grid_deg=np.arange(0,90,3))
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(P, aspect='auto', origin='lower',
               extent=(az_grid[0], az_grid[-1], el_grid[0], el_grid[-1]),
               cmap='viridis')
fig.colorbar(im, ax=ax, label='P(az, el) (a.u.)')
ax.plot(az_deg, el_deg, 'rx', ms=12, mew=2, label='initial guess')
i, j = np.unravel_index(np.argmax(P), P.shape)
ax.plot(az_grid[j], el_grid[i], 'wo', ms=10, mfc='none', mew=2,
        label=f'best fit ({az_grid[j]:.0f}°, {el_grid[i]:.0f}°)')
ax.set_xlabel('azimuth (deg, N→E)'); ax.set_ylabel('elevation (deg)')
ax.set_title('Beamforming: P(az, el) = ⟨|B_⊥|²⟩')
ax.legend(); fig.tight_layout()